# Model Inference: SQL & REST API

**Session 4 — Deep Dive: Deployment, Inference & MLOps**  
**Notebook 2 of 3** | ~15 minutes

---

## What You Will Learn

| Concept | What We Cover |
|---------|---------------|
| **Model Aliases** | Label versions (alpha, beta, production) for lifecycle management |
| **Default Version** | Set the default version for simple SQL inference |
| **SQL Inference** | Call models directly in SQL queries using `MODEL!predict()` |
| **REST Inference** | Call the deployed service via HTTP using the `requests` library |
| **Batch vs Real-time** | Understand when to use each inference approach |

---

## Inference Approaches in Snowflake

| Approach | Compute | Latency | Best For |
|----------|---------|---------|----------|
| **SQL (Warehouse)** | Virtual Warehouse | Medium | Batch scoring, pipelines, dbt |
| **REST (SPCS Service)** | Compute Pool | Low | Real-time apps, microservices |
| **REST (Gateway)** | Compute Pool | Low | Production: stable URL + traffic split |

## 1 | Environment Setup

In [ ]:
%cd ..
%load_ext autoreload

In [ ]:
import json
import os

import pandas as pd
import requests
from snowflake.ml.registry import Registry

from utils import get_config
from utils import get_session

config = get_config("config.yaml")
DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
WH = config.snowflake.warehouse
MODEL_NAME = config.model.model_name
SERVICE_NAME = config.deploy.service_name

session = get_session(
    config.snowflake.connection_name,
    database_name=DB,
    schema_name=SCHEMA,
    warehouse_name=WH,
)
print(f"Connected: {session.get_current_database()}.{session.get_current_schema()}")

In [ ]:
reg = Registry(session=session, database_name=DB, schema_name=SCHEMA)
model = reg.get_model(MODEL_NAME)

print(f"Model: {MODEL_NAME}")
print(f"Versions: {[v.version_name for v in model.versions()]}")
print(f"Default version: {model.default.version_name}")

## 2 | Model Aliases & Lifecycle Management

Aliases are user-defined labels attached to model versions for lifecycle tracking.
Common patterns:

```
 V1 ─── alias: "production"    (currently serving traffic)
 V2 ─── alias: "staging"       (under validation)
 V3 ─── alias: "development"   (latest experimental)
```

Aliases are **exclusive** — only one version can hold a given alias at a time.

In [ ]:
mv = model.default

session.sql(f"""
    ALTER MODEL {DB}.{SCHEMA}.{MODEL_NAME}
    VERSION {mv.version_name}
    SET ALIAS = production
""").collect()
print(f"Set alias 'production' on version: {mv.version_name}")

In [ ]:
session.sql(f"SHOW VERSIONS IN MODEL {DB}.{SCHEMA}.{MODEL_NAME}").show()

## 3 | Set Default Version for SQL Inference

The **default version** is what gets invoked when you call the model without specifying a version.
This is the simplest production pattern:

```sql
-- Uses the default version automatically
SELECT my_model!predict(...) FROM my_table;

-- Uses a specific version by alias
WITH v AS MODEL my_model VERSION production
    SELECT v!predict(...) FROM my_table;
```

In [ ]:
session.sql(f"""
    ALTER MODEL {DB}.{SCHEMA}.{MODEL_NAME}
    SET DEFAULT_VERSION = {mv.version_name}
""").collect()
print(f"Default version set to: {mv.version_name}")

## 4 | SQL Inference (Warehouse-based)

Call the model directly in SQL — runs on your virtual warehouse.  
No service required, no REST — just SQL.

In [ ]:
result = session.sql(f"""
    SELECT
        PATIENT_ID,
        {MODEL_NAME}!PREDICT(
            AGE, BMI, HEART_RATE, SYSTOLIC_BP, DIASTOLIC_BP,
            TEMPERATURE, RESPIRATORY_RATE, OXYGEN_SATURATION,
            GLUCOSE_LEVEL, CREATININE, HEMOGLOBIN, WBC_COUNT,
            COMORBIDITY_COUNT, PREVIOUS_ADMISSIONS, MEDICATION_COUNT,
            GENDER, PRIMARY_DIAGNOSIS, ADMISSION_TYPE, INSURANCE_TYPE,
            SHOCK_INDEX, PULSE_PRESSURE, BMI_CATEGORY, VITAL_SIGNS_SEVERITY
        ) AS PREDICTION
    FROM {DB}.{SCHEMA}.{config.tables.raw_data}
    LIMIT 5
""").collect()

print("SQL Inference (default version):")
for row in result:
    print(f"  Patient {row['PATIENT_ID']}: {row['PREDICTION']}")

In [ ]:
result = session.sql(f"""
    WITH model_v AS MODEL {DB}.{SCHEMA}.{MODEL_NAME} VERSION production
    SELECT
        PATIENT_ID,
        model_v!PREDICT(
            AGE, BMI, HEART_RATE, SYSTOLIC_BP, DIASTOLIC_BP,
            TEMPERATURE, RESPIRATORY_RATE, OXYGEN_SATURATION,
            GLUCOSE_LEVEL, CREATININE, HEMOGLOBIN, WBC_COUNT,
            COMORBIDITY_COUNT, PREVIOUS_ADMISSIONS, MEDICATION_COUNT,
            GENDER, PRIMARY_DIAGNOSIS, ADMISSION_TYPE, INSURANCE_TYPE,
            SHOCK_INDEX, PULSE_PRESSURE, BMI_CATEGORY, VITAL_SIGNS_SEVERITY
        ) AS PREDICTION
    FROM {DB}.{SCHEMA}.{config.tables.raw_data}
    LIMIT 5
""").collect()

print("SQL Inference (via 'production' alias):")
for row in result:
    print(f"  Patient {row['PATIENT_ID']}: {row['PREDICTION']}")

## 5 | Batch Inference Pattern

For scoring an entire table (e.g., nightly batch), use a CTAS or INSERT:

```sql
CREATE OR REPLACE TABLE predictions AS
SELECT patient_id, model!predict(...) AS prediction
FROM features_table;
```

In [ ]:
session.sql(f"""
    CREATE OR REPLACE TABLE {DB}.{SCHEMA}.BATCH_PREDICTIONS AS
    SELECT
        PATIENT_ID,
        ENCOUNTER_ID,
        CURRENT_TIMESTAMP() AS SCORED_AT,
        {MODEL_NAME}!PREDICT(
            AGE, BMI, HEART_RATE, SYSTOLIC_BP, DIASTOLIC_BP,
            TEMPERATURE, RESPIRATORY_RATE, OXYGEN_SATURATION,
            GLUCOSE_LEVEL, CREATININE, HEMOGLOBIN, WBC_COUNT,
            COMORBIDITY_COUNT, PREVIOUS_ADMISSIONS, MEDICATION_COUNT,
            GENDER, PRIMARY_DIAGNOSIS, ADMISSION_TYPE, INSURANCE_TYPE,
            SHOCK_INDEX, PULSE_PRESSURE, BMI_CATEGORY, VITAL_SIGNS_SEVERITY
        ) AS PREDICTION
    FROM {DB}.{SCHEMA}.{config.tables.raw_data}
""").collect()

count = session.sql(f"SELECT COUNT(*) AS N FROM {DB}.{SCHEMA}.BATCH_PREDICTIONS").collect()[0]["N"]
print(f"Batch predictions created: {count} rows")
session.sql(f"SELECT * FROM {DB}.{SCHEMA}.BATCH_PREDICTIONS LIMIT 5").show()

## 6 | REST API Inference (Real-time)

For real-time inference from external apps, call the service HTTP endpoint.

### Two Calling Patterns

| From | Endpoint | Auth |
|------|----------|------|
| **Inside Snowflake** (notebooks, UDFs) | Internal endpoint | None needed |
| **External** (apps, scripts) | Public ingress URL | PAT token |

### Request Format: `dataframe_split` (Recommended)

```json
{
  "dataframe_split": {
    "index": [0],
    "columns": ["feature1", "feature2", ...],
    "data": [[val1, val2, ...]]
  }
}
```

In [ ]:
services_df = mv.list_services()
print(services_df[["name", "status", "inference_endpoint", "internal_endpoint"]].to_string())

inference_endpoint = services_df[services_df["name"] == SERVICE_NAME]["inference_endpoint"].iloc[0]

internal_endpoint = services_df[services_df["name"] == SERVICE_NAME]["internal_endpoint"].iloc[0]

print(f"\nPublic endpoint:   {inference_endpoint}")
print(f"Internal endpoint: {internal_endpoint}")

In [ ]:
sample_row = session.sql(f"""
    SELECT AGE, BMI, HEART_RATE, SYSTOLIC_BP, DIASTOLIC_BP,
           TEMPERATURE, RESPIRATORY_RATE, OXYGEN_SATURATION,
           GLUCOSE_LEVEL, CREATININE, HEMOGLOBIN, WBC_COUNT,
           COMORBIDITY_COUNT, PREVIOUS_ADMISSIONS, MEDICATION_COUNT,
           GENDER, PRIMARY_DIAGNOSIS, ADMISSION_TYPE, INSURANCE_TYPE,
           SHOCK_INDEX, PULSE_PRESSURE, BMI_CATEGORY, VITAL_SIGNS_SEVERITY
    FROM {DB}.{SCHEMA}.{config.tables.raw_data}
    LIMIT 1
""").to_pandas()

print("Sample input for REST call:")
print(sample_row.to_string(index=False))

In [ ]:
url = f"{internal_endpoint}/predict"

payload = json.loads(sample_row.to_json(orient="split"))
request_body = {"dataframe_split": payload}

print(f"POST {url}")
print(f"Payload (truncated): {json.dumps(request_body, indent=2)[:500]}...")

response = requests.post(url, json=request_body, timeout=30)

print(f"\nStatus: {response.status_code}")
print(f"Response: {json.dumps(response.json(), indent=2)}")

## 7 | REST API from External Clients (PAT Auth)

For calling from outside Snowflake, use a **Programmatic Access Token (PAT)**:

1. Go to Snowsight → User Menu → Settings → Authentication
2. Create a new Programmatic Access Token
3. Pass it in the `Authorization` header

> **Note:** In a URL, underscores in method names become dashes: `predict_proba` → `/predict-proba`

In [ ]:
# PAT = "your-pat-token-here"  # Set your PAT token
# endpoint_url = f"https://{inference_endpoint}/predict"
#
# headers = {
#     "Authorization": f'Snowflake Token="{PAT}"',
#     "Content-Type": "application/json",
# }
#
# payload = json.loads(sample_row.to_json(orient="split"))
# request_body = {"dataframe_split": payload}
#
# response = requests.post(endpoint_url, json=request_body, headers=headers, timeout=30)
# print(f"Status: {response.status_code}")
# print(f"Response: {json.dumps(response.json(), indent=2)}")

print("Uncomment and set your PAT token to test external REST API calls.")
print(f"Endpoint: https://{inference_endpoint}/predict")

## 8 | Inference via Gateway

The Gateway provides a **stable URL** that doesn't change when you recreate services.  
This is the recommended approach for production REST integrations.

In [ ]:
GATEWAY_NAME = "PATIENT_RISK_GATEWAY"

gw_info = session.sql(f"DESC GATEWAY {DB}.{SCHEMA}.{GATEWAY_NAME}").collect()
gw_endpoint = None
for row in gw_info:
    if "ingress_url" in str(row):
        gw_endpoint = row["ingress_url"] if "ingress_url" in row.asDict() else None

print(f"Gateway: {GATEWAY_NAME}")
print(f"Endpoint: {gw_endpoint}")
print("\nThe gateway URL remains stable even if the underlying service is recreated.")

## Summary

In this notebook we covered:

1. **Aliases** — Label model versions with lifecycle stages (`production`, `staging`, etc.)
2. **Default Version** — Set which version is used for bare `MODEL!predict()` SQL calls
3. **SQL Inference** — Call models directly in SQL (warehouse-based, no service needed)
4. **Batch Inference** — Score entire tables with CTAS pattern
5. **REST Inference** — Call the SPCS service via HTTP using `dataframe_split` format
6. **Gateway Inference** — Use the stable gateway URL for production REST integrations

---

**Next:** [03_model_monitoring.ipynb](./03_model_monitoring.ipynb) — Set up model monitoring, drift detection, and automated re-training alerts